# HydroSAR-BD: Reviewer Revisions Replication Notebook

This Jupyter Notebook provides the complete, end-to-end code required to reproduce the statistical and visual validations requested by the reviewers for the **HydroSAR-BD** manuscript.

## Notebook Structure
1. **Task 3:** GMM Component Justification using Akaike/Bayesian Information Criterion (AIC/BIC) tests.
2. **Task 2:** Per-class accuracy assessment (User's and Producer's Accuracy) across different hydrological permanence classes (Permanent, Semi-permanent, and Ephemeral).
3. **Task 1:** Visual 5-panel comparative mapping (SAR VV, Random Forest, Otsu, ST-GMM, and NDWI).

### Running in Google Colab
If running in Google Colab, upload your datasets (`data/` folder) to your Google Drive or upload them directly to the Colab environment using the folder icon on the left.

In [ ]:
# Imports and Configuration
import os
import ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
import warnings
warnings.filterwarnings('ignore')

# Try installing rasterio if not present (required for Task 1 map plotting)
try:
    import rasterio
except ImportError:
    print("Installing rasterio...")
    !pip install -q rasterio
    import rasterio

## Section 1: Task 3 - GMM Component Justification (AIC/BIC Test)

This section fits Gaussian Mixture Models (GMM) with 2, 3, 4, and 5 components to Sentinel-1 backscatter distributions extracted from three diverse geographical districts in Bangladesh (Sunamganj - deep haor wetlands, Dhaka - urban/industrial, Bhola - coastal island).

Lower AIC/BIC scores indicate a better balance between model fit and complexity (preventing overfitting).

In [ ]:
# Path to the district-wise histograms CSV
hist_csv_path = "data/Bangladesh_District_VV_Histograms_2015_2025.csv"

if not os.path.exists(hist_csv_path):
    print(f"[ERROR] File {hist_csv_path} not found. Please upload/provide the file.")
else:
    print("Loading histogram dataset...")
    df_hist = pd.read_csv(hist_csv_path)
    
    # Parse list strings
    df_hist['histogram_counts'] = df_hist['histogram_counts'].apply(ast.literal_eval)
    if 'histogram_means' in df_hist.columns:
        df_hist['histogram_means'] = df_hist['histogram_means'].apply(ast.literal_eval)
    else:
        df_hist['histogram_means'] = [np.linspace(-30, 5, len(h)) for h in df_hist['histogram_counts']]

    sample_districts = ['Sunamganj', 'Dhaka', 'Bhola']
    results = []
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for idx, dist in enumerate(sample_districts):
        # Filter for the district in August (monsoon peak)
        subset = df_hist[(df_hist['district_name'] == dist) & (df_hist['month'] == 8)]
        if subset.empty:
            subset = df_hist[df_hist['district_name'] == dist]
            
        row = subset.iloc[0]
        counts = np.array(row['histogram_counts'])
        bins = np.array(row['histogram_means'])
        
        # Reconstruct 1D samples
        mask = counts > 0
        counts = counts[mask]
        bins = bins[mask]
        
        total_counts = counts.sum()
        scale_factor = max(1, total_counts // 100000)
        scaled_counts = (counts / scale_factor).astype(int)
        samples = np.repeat(bins, scaled_counts).reshape(-1, 1)
        
        aic_scores = []
        bic_scores = []
        n_components_range = [2, 3, 4, 5]
        
        print(f"Fitting GMM components for {dist}...")
        for n in n_components_range:
            gmm = GaussianMixture(n_components=n, covariance_type='full', max_iter=200, random_state=42)
            gmm.fit(samples)
            
            results.append({
                'District': dist,
                'Components': n,
                'AIC': gmm.aic(samples),
                'BIC': gmm.bic(samples)
            })
            aic_scores.append(gmm.aic(samples))
            bic_scores.append(gmm.bic(samples))
            
        ax = axes[idx]
        ax.plot(n_components_range, aic_scores, marker='o', label='AIC', linewidth=2)
        ax.plot(n_components_range, bic_scores, marker='s', label='BIC', linewidth=2)
        ax.set_title(f"{dist} (District)")
        ax.set_xlabel("Number of GMM Components")
        ax.set_ylabel("Information Criterion Score")
        ax.set_xticks(n_components_range)
        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.5)
        
    plt.suptitle("GMM Goodness-of-Fit Diagnostic (AIC/BIC)", fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Save and display scores
    results_df = pd.DataFrame(results)
    print("\nAIC / BIC Scores Summary Table:")
    print(results_df.to_string(index=False))

## Section 2: Task 2 - Per-Class Accuracy Assessment

Here we calculate User's Accuracy (UA) and Producer's Accuracy (PA) across the three water hydroperiod permanence classes: 
- **Permanent Water:** occurrence frequency $\ge 80\%$
- **Semi-Permanent Water:** occurrence frequency $40\% < F < 80\%$
- **Ephemeral Water:** occurrence frequency $0\% < F \le 40\%$

These frequencies are extracted using the JRC Global Surface Water Occurrence dataset.

In [ ]:
# Path to validation points with JRC occurrence data
validation_csv_path = "data/Validation_Points_With_Occurrence.csv"

if not os.path.exists(validation_csv_path):
    print(f"[ERROR] File {validation_csv_path} not found.")
    print("Please run 'gee_scripts/task2_extract_occurrence.js' in GEE and upload the resulting CSV to the 'data/' folder.")
else:
    df_val = pd.read_csv(validation_csv_path)
    df_val['occurrence'] = df_val['occurrence'].fillna(0)
    
    def classify_occurrence(occ):
        if occ >= 80:
            return 'Permanent'
        elif 40 < occ < 80:
            return 'Semi-permanent'
        elif 0 < occ <= 40:
            return 'Ephemeral'
        else:
            return 'Non-Water'
            
    df_val['Water_Class'] = df_val['occurrence'].apply(classify_occurrence)
    
    results = []
    classes = ['Permanent', 'Semi-permanent', 'Ephemeral']
    
    for cls in classes:
        sub_df = df_val[df_val['Water_Class'] == cls]
        if len(sub_df) == 0: 
            continue
            
        y_true = sub_df['Field_Truth']
        y_pred = sub_df['class']  # Predicted column from model
        
        tp = ((y_true == 1) & (y_pred == 1)).sum()
        fp = ((y_true == 0) & (y_pred == 1)).sum()
        fn = ((y_true == 1) & (y_pred == 0)).sum()
        tn = ((y_true == 0) & (y_pred == 0)).sum()
        
        ua = (tp / (tp + fp)) * 100 if (tp + fp) > 0 else 0.0
        pa = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0.0
        oa = ((tp + tn) / len(sub_df)) * 100
        
        results.append({
            'Water Class': cls,
            'Total Samples': len(sub_df),
            'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
            'User Accuracy (UA %)': round(ua, 2),
            'Producer Accuracy (PA %)': round(pa, 2),
            'Overall Accuracy (OA %)': round(oa, 2)
        })
        
    results_df = pd.DataFrame(results)
    print("\n" + "="*70)
    print("             PER-CLASS ACCURACY METRICS TABLE")
    print("="*70)
    print(results_df.to_string(index=False))
    print("="*70)

## Section 3: Task 1 - 5-Panel Comparative Map

This section maps the 5 comparative layers side-by-side using the exported GeoTIFFs for Gazipur District (September 2020).

In [ ]:
raster_dir = "data/Task1_Rasters"
files = {
    'SAR_VV': os.path.join(raster_dir, 'Task1_1_SAR_VV.tif'),
    'RF': os.path.join(raster_dir, 'Task1_2_RandomForest.tif'),
    'Otsu': os.path.join(raster_dir, 'Task1_3_Otsu.tif'),
    'ST-GMM': os.path.join(raster_dir, 'Task1_4_ST_GMM.tif'),
    'NDWI': os.path.join(raster_dir, 'Task1_5_NDWI.tif')
}

missing_files = [f for f in files.values() if not os.path.exists(f)]
if missing_files:
    print(f"[ERROR] The following rasters are missing: {missing_files}")
    print("Please run 'gee_scripts/task1_export_5panels.js' in GEE, download the results, and place them in 'data/Task1_Rasters/'.")
else:
    fig, axes = plt.subplots(1, 5, figsize=(25, 6))
    titles = [
        "(a) Sentinel-1 SAR VV\n(Raw Backscatter)", 
        "(b) Random Forest\n(Supervised)", 
        "(c) Otsu Thresholding\n(Global Unsupervised)", 
        "(d) ST-GMM\n(Proposed Method)", 
        "(e) Sentinel-2 NDWI\n(Optical Reference)"
    ]
    
    cmap_binary = plt.colors.ListedColormap(['#e0e0e0', '#004c99'])
    keys = ['SAR_VV', 'RF', 'Otsu', 'ST-GMM', 'NDWI']
    
    for i, key in enumerate(keys):
        ax = axes[i]
        with rasterio.open(files[key]) as src:
            img = src.read(1)
            img = np.ma.masked_where(img < -9999, img)
            
            if key == 'SAR_VV':
                ax.imshow(img, cmap='gray', vmin=-25, vmax=0)
            else:
                ax.imshow(img, cmap=cmap_binary, vmin=0, vmax=1)
                
        ax.set_title(titles[i], fontsize=12, pad=12, fontweight='bold')
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()